# Email Campaign Experiment
Men's email vs women's email vs no-email control (64k users). Questions: which campaign wins, where in the funnel, how much extra revenue, is it statistically real, and **which customers should get which email?**

Run `sql/01_clean_table.sql` once, then this notebook. All aggregation happens in BigQuery (`sql/02_summary.sql` overall, `sql/03_segments.sql` by customer segment), so the notebook works from small summary tables instead of 64k rows.

> Causal wording assumes random assignment, which the balance check below tests.

In [ ]:
import pandas as pd
from google.cloud import bigquery
from scipy.stats import ttest_ind_from_stats, chi2_contingency
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

bq = bigquery.Client(project="abtes-510522")
s = bq.query(open("sql/02_summary.sql").read()).to_dataframe().set_index("variant_group")
assert s.bad_rows.sum() == 0 and s.neg_spend.sum() == 0, "data check failed"
s

In [ ]:
# funnel KPIs
k = pd.DataFrame({
    "users": s.users,
    "ctr": s.visits / s.users,
    "post_click_conv": s.conversions / s.visits,
    "conv_rate": s.conversions / s.users,
    "rev_per_user": s.rev_per_user,
    "rev_per_converter": s.revenue / s.conversions,
    "net_lift_$": (s.rev_per_user - s.rev_per_user["no e-mail"]) * s.users,
})
k

## Was assignment random?
If the segment mix differs across groups, the lift could be a mix effect rather than an email effect. Chi-square p-values below; small values (well under 0.05) would be a red flag.

In [ ]:
seg = bq.query(open("sql/03_segments.sql").read()).to_dataframe()

balance_p = {dim: chi2_contingency(g.pivot(index="segment", columns="variant_group", values="users").fillna(0))[1]
             for dim, g in seg.groupby("dimension")}
pd.Series(balance_p, name="chi2_p").round(3)

In [ ]:
# pairwise tests, Holm-adjusted per metric family
def test(a, b):
    ra, rb = s.loc[a], s.loc[b]
    return {
        "pair": f"{a} vs {b}",
        "ctr_p": proportions_ztest([ra.visits, rb.visits], [ra.users, rb.users])[1],
        "conv_p": proportions_ztest([ra.conversions, rb.conversions], [ra.users, rb.users])[1],
        "post_click_p": proportions_ztest([ra.conversions, rb.conversions], [ra.visits, rb.visits])[1],
        "revenue_p": ttest_ind_from_stats(ra.rev_per_user, ra.rev_std, ra.users,
                                          rb.rev_per_user, rb.rev_std, rb.users, equal_var=False).pvalue,
    }

pairs = [("mens e-mail", "no e-mail"), ("womens e-mail", "no e-mail"), ("mens e-mail", "womens e-mail")]
p = pd.DataFrame([test(a, b) for a, b in pairs]).set_index("pair")
for c in p:
    p[c] = multipletests(p[c], method="holm")[1]
p.apply(lambda c: c.map("{:.2e}".format))

## How big is the lift, with uncertainty?

In [ ]:
def lift_ci(m1, sd1, n1, m0, sd0, n0):
    """Difference in mean revenue per user with a 95% CI (normal approx; n is ~21k per group)."""
    d = m1 - m0
    se = (sd1**2 / n1 + sd0**2 / n0) ** 0.5
    return d, d - 1.96 * se, d + 1.96 * se

c = s.loc["no e-mail"]
rows = {}
for e in ["mens e-mail", "womens e-mail"]:
    r = s.loc[e]
    d, lo, hi = lift_ci(r.rev_per_user, r.rev_std, r.users, c.rev_per_user, c.rev_std, c.users)
    rows[e] = {"lift_per_user": d, "ci_low": lo, "ci_high": hi,
               "incremental_$": d * r.users, "incr_low_$": lo * r.users, "incr_high_$": hi * r.users}
pd.DataFrame(rows).T.round(2)

In [ ]:
# charts
names = {"no e-mail": "Control", "womens e-mail": "Women's", "mens e-mail": "Men's"}
order = list(names)
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, col, title, fmt in zip(ax, ["ctr", "conv_rate", "rev_per_user"],
                              ["Click-through rate", "Conversion rate", "Revenue per user"],
                              ["{:.1%}", "{:.2%}", "${:.2f}"]):
    v = k.loc[order, col]
    bars = a.bar([names[i] for i in order], v)
    a.bar_label(bars, labels=[fmt.format(x) for x in v], padding=3)
    a.set_title(title)
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

## Who responds? Revenue lift per user by segment
Each dot is the email's lift vs control within a customer segment, with a 95% CI. Bars crossing zero are not distinguishable from no effect.

In [ ]:
w = seg.pivot_table(index=["dimension", "segment"], columns="variant_group",
                    values=["users", "rev_per_user", "rev_std"])

res = {}
for e in ["mens e-mail", "womens e-mail"]:
    d, lo, hi = lift_ci(w["rev_per_user"][e], w["rev_std"][e], w["users"][e],
                        w["rev_per_user"]["no e-mail"], w["rev_std"]["no e-mail"], w["users"]["no e-mail"])
    res[e] = pd.DataFrame({"lift": d, "lo": lo, "hi": hi}).dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 8))
for ax, (e, r) in zip(axes, res.items()):
    ax.errorbar(r.lift, range(len(r)), xerr=[r.lift - r.lo, r.hi - r.lift], fmt="o", capsize=3)
    ax.axvline(0, color="grey", lw=1)
    ax.set_yticks(range(len(r)), [f"{a}: {b}" for a, b in r.index])
    ax.invert_yaxis()
    ax.set_title(f"{names[e]} email: lift per user ($)")
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

## Who should get which email?
Simple rule on one segment cut: send each segment whichever email lifts revenue more, and skip a segment if the lift doesn't cover the cost of sending. `COST` is an assumption; replace it with the real cost per email.

In [ ]:
COST = 0.10          # assumed $ cost per email send
DIM = "past_purchase"

u = w["users"].xs(DIM)
t = pd.DataFrame({
    "share": u.sum(axis=1) / u.sum().sum(),
    "mens": res["mens e-mail"].lift.xs(DIM),
    "womens": res["womens e-mail"].lift.xs(DIM),
})
t["best"] = t[["mens", "womens"]].idxmax(axis=1)
t["best_net"] = t[["mens", "womens"]].max(axis=1) - COST

display(t.round(3))

profit_per_1000 = pd.Series({
    "email everyone: men's": 1000 * (t.share * (t.mens - COST)).sum(),
    "email everyone: women's": 1000 * (t.share * (t.womens - COST)).sum(),
    "targeted: best email per segment, skip if lift < cost": 1000 * (t.share * t.best_net.clip(lower=0)).sum(),
}).round(0)
profit_per_1000.rename("incremental revenue minus email cost, per 1,000 customers")

Caveat: segment lifts are estimated on the same data used to pick the targets, so the targeted number is optimistic. Confirm with a holdout test before rolling out.

## Takeaways
- **Men's email wins on CTR, conversion rate and revenue per user**: CTR 18.3% vs 15.1% (women's) vs 10.6% (control); conversion 1.25% / 0.88% / 0.57%; revenue per user $1.42 / $1.08 / $0.65.
- Estimated incremental revenue vs control: **~$16.4k men's, ~$9.1k women's** (~$25.5k combined).
- The lift comes mostly from **getting more people to click**. Post-click conversion (6.9% / 5.8% / 5.4%) and revenue per converter (~$114-122) are not significantly different across groups after Holm correction, so don't claim the men's email makes visitors buy more.
- CTR, conversion rate and revenue lifts vs control are all significant after Holm correction. Men's vs women's revenue is the weakest (p ≈ 0.03).
- Spend is heavily zero-inflated (max $499); with ~21k users per group the CI on mean lift is still reasonable, but treat the t-test p-values as approximate.

### Segment findings (fill in after running)
- TODO balance: <did any chi-square p fall well below 0.05?>
- TODO who responds: <which segments show a clear lift, which cross zero?>
- TODO targeting: <targeted vs email-everyone, per 1,000 customers, at your cost assumption>

Next: validate the targeting rule with a holdout test; investigate why the women's creative underperformed.